#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Loading the Dataset

In [3]:
df = pd.read_csv("../data/raw/Hotel Reservations.csv")

print("Dataset Shape:", df.shape)
print("\nDataset Preview:")
print(df.head().to_string(index=False))

Dataset Shape: (36275, 19)

Dataset Preview:
Booking_ID  no_of_adults  no_of_children  no_of_weekend_nights  no_of_week_nights type_of_meal_plan  required_car_parking_space room_type_reserved  lead_time  arrival_year  arrival_month  arrival_date market_segment_type  repeated_guest  no_of_previous_cancellations  no_of_previous_bookings_not_canceled  avg_price_per_room  no_of_special_requests booking_status
  INN00001             2               0                     1                  2       Meal Plan 1                           0        Room_Type 1        224          2017             10             2             Offline               0                             0                                     0               65.00                       0   Not_Canceled
  INN00002             2               0                     2                  3      Not Selected                           0        Room_Type 1          5          2018             11             6              Online       

#### Creating the Processed Data Directory

In [4]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Processed Data Directory: {processed_dir.resolve()}")
print(f"Directory Exists: {processed_dir.exists()}")

Processed Data Directory: D:\Projects\ML and Data Science\StaySense Analytics\data\processed
Directory Exists: True


#### Validating Dataset Quality

In [5]:
print("Dataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().to_string())

print("\nFully Duplicated Records:")
print(df.duplicated().sum())

print("\nDuplicated Booking IDs:")
print(df["Booking_ID"].duplicated().sum())

print("\nColumn Data Types:")
print(df.dtypes.to_string())

print("\nBooking Status Distribution:")
print(df["booking_status"].value_counts().to_string())

Dataset Shape:
(36275, 19)

Missing Values:
Booking_ID                              0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0

Fully Duplicated Records:
0

Duplicated Booking IDs:
0

Column Data Types:
Booking_ID                                  str
no_of_adults                      

#### Key Observations
- **Complete Dataset:** The dataset contains 36,275 reservations across 19 columns, with no missing values in any feature.
- **Unique Reservation Records:** No fully duplicated records or duplicated booking identifiers were identified, confirming that each reservation has a unique Booking_ID.
- **Consistent Data Types:** Numerical and categorical variables have appropriate data types, while booking outcomes comprise 24,390 non-canceled reservations and 11,885 canceled reservations.

#### Validating Scheduled Arrival Dates

In [6]:
df["arrival_datetime"] = pd.to_datetime(
    df[["arrival_year", "arrival_month", "arrival_date"]].rename(
        columns={
            "arrival_year": "year",
            "arrival_month": "month",
            "arrival_date": "day"
        }
    ),
    errors="coerce"
)

invalid_arrivals = df[df["arrival_datetime"].isna()]

print("Arrival Date Validation Summary:")
print(f"Total Reservations: {len(df):,}")
print(f"Valid Arrival Dates: {df['arrival_datetime'].notna().sum():,}")
print(f"Invalid Arrival Dates: {len(invalid_arrivals):,}")

print("\nInvalid Arrival Date Combinations:")
print(
    invalid_arrivals[
        ["arrival_year", "arrival_month", "arrival_date"]
    ]
    .value_counts()
    .to_string()
)

print("\nValid Arrival Date Range:")
print(f"Start Date: {df['arrival_datetime'].min().date()}")
print(f"End Date: {df['arrival_datetime'].max().date()}")

Arrival Date Validation Summary:
Total Reservations: 36,275
Valid Arrival Dates: 36,238
Invalid Arrival Dates: 37

Invalid Arrival Date Combinations:
arrival_year  arrival_month  arrival_date
2018          2              29              37

Valid Arrival Date Range:
Start Date: 2017-07-01
End Date: 2018-12-31


#### Key Observations
- **Invalid Arrival Dates:** The dataset contains 37 reservations (0.10%) with invalid scheduled arrival dates, all corresponding to February 29, 2018.
- **Valid Temporal Coverage:** The remaining 36,238 reservations have valid arrival dates spanning July 1, 2017, through December 31, 2018.
- **Data Preparation Decision:** Invalid-date records will be excluded from the forecasting dataset because their actual scheduled arrival dates cannot be reliably determined.

#### Cleaning and Saving Reservation-Level Data

In [7]:
df_clean = (
    df.dropna(subset=["arrival_datetime"])
    .copy()
    .sort_values(["arrival_datetime", "Booking_ID"])
    .reset_index(drop=True)
)

cleaned_file = processed_dir / "cleaned_reservations.csv"

df_clean.to_csv(cleaned_file, index=False)

print("Cleaned Reservation Dataset:")
print(f"Original Records: {len(df):,}")
print(f"Removed Invalid-Date Records: {len(df) - len(df_clean):,}")
print(f"Cleaned Records: {len(df_clean):,}")
print(f"Dataset Shape: {df_clean.shape}")

print("\nValidation:")
print(f"Missing Arrival Dates: {df_clean['arrival_datetime'].isna().sum():,}")
print(f"Duplicated Booking IDs: {df_clean['Booking_ID'].duplicated().sum():,}")
print(f"Chronologically Sorted: {df_clean['arrival_datetime'].is_monotonic_increasing}")

print("\nSaved Artifact:")
print(f"File: {cleaned_file}")
print(f"File Exists: {cleaned_file.exists()}")

Cleaned Reservation Dataset:
Original Records: 36,275
Removed Invalid-Date Records: 37
Cleaned Records: 36,238
Dataset Shape: (36238, 20)

Validation:
Missing Arrival Dates: 0
Duplicated Booking IDs: 0
Chronologically Sorted: True

Saved Artifact:
File: ..\data\processed\cleaned_reservations.csv
File Exists: True


#### Key Observations
- **Successful Data Cleaning:** Removed 37 reservations with invalid arrival dates, retaining 36,238 valid records.
- **Data Integrity Preserved:** The cleaned dataset contains no missing arrival dates or duplicated booking identifiers, and all records are chronologically sorted.
- **Reusable Artifact Saved:** The cleaned reservation-level dataset has been saved as `data/processed/cleaned_reservations.csv` for subsequent analysis.

#### Aggregating Daily Scheduled Reservations

In [8]:
daily_df = (
    df_clean.groupby("arrival_datetime")
    .agg(
        scheduled_reservations=("Booking_ID", "count")
    )
    .sort_index()
    .asfreq("D", fill_value=0)
    .reset_index()
)

print("Daily Forecasting Dataset:")
print(f"Dataset Shape: {daily_df.shape}")
print(f"Start Date: {daily_df['arrival_datetime'].min().date()}")
print(f"End Date: {daily_df['arrival_datetime'].max().date()}")

print("\nReservation Count Validation:")
print(f"Cleaned Reservations: {len(df_clean):,}")
print(f"Aggregated Reservations: {daily_df['scheduled_reservations'].sum():,}")
print(f"Counts Match: {daily_df['scheduled_reservations'].sum() == len(df_clean)}")

print("\nDaily Reservation Summary:")
print(daily_df["scheduled_reservations"].describe().round(2).to_string())

print("\nDataset Preview:")
print(daily_df.head(10).to_string(index=False))

Daily Forecasting Dataset:
Dataset Shape: (549, 2)
Start Date: 2017-07-01
End Date: 2018-12-31

Reservation Count Validation:
Cleaned Reservations: 36,238
Aggregated Reservations: 36,238
Counts Match: True

Daily Reservation Summary:
count    549.00
mean      66.01
std       43.56
min        1.00
25%       32.00
50%       61.00
75%       92.00
max      254.00

Dataset Preview:
arrival_datetime  scheduled_reservations
      2017-07-01                      68
      2017-07-02                       5
      2017-07-03                       3
      2017-07-04                       4
      2017-07-05                       3
      2017-07-06                      10
      2017-07-07                      14
      2017-07-08                       7
      2017-07-09                       4
      2017-07-10                       5


#### Key Observations
- **Successful Daily Aggregation:** The 36,238 valid reservation records have been aggregated into 549 daily observations spanning July 1, 2017, through December 31, 2018.
- **Reservation Count Consistency:** The aggregated reservation total matches the cleaned dataset exactly, confirming that no valid reservations were lost during aggregation.
- **Forecasting Target Distribution:** Daily scheduled reservations average 66.01, with a standard deviation of 43.56 and values ranging from 1 to 254.

#### Validating Temporal Continuity and Target Integrity

In [9]:
expected_dates = pd.date_range(
    start=daily_df["arrival_datetime"].min(),
    end=daily_df["arrival_datetime"].max(),
    freq="D"
)

missing_dates = expected_dates.difference(
    pd.DatetimeIndex(daily_df["arrival_datetime"])
)

print("Temporal Continuity Validation:")
print(f"Expected Calendar Days: {len(expected_dates):,}")
print(f"Observed Calendar Days: {len(daily_df):,}")
print(f"Missing Calendar Dates: {len(missing_dates):,}")
print(f"Duplicated Dates: {daily_df['arrival_datetime'].duplicated().sum():,}")
print(f"Chronologically Sorted: {daily_df['arrival_datetime'].is_monotonic_increasing}")

print("\nTarget Integrity Validation:")
print(f"Missing Target Values: {daily_df['scheduled_reservations'].isna().sum():,}")
print(f"Negative Target Values: {(daily_df['scheduled_reservations'] < 0).sum():,}")
print(f"Zero-Reservation Days: {(daily_df['scheduled_reservations'] == 0).sum():,}")
print(f"Target Data Type: {daily_df['scheduled_reservations'].dtype}")

print("\nOverall Validation:")
validation_passed = (
    len(missing_dates) == 0
    and not daily_df["arrival_datetime"].duplicated().any()
    and daily_df["arrival_datetime"].is_monotonic_increasing
    and daily_df["scheduled_reservations"].notna().all()
    and (daily_df["scheduled_reservations"] >= 0).all()
    and daily_df["scheduled_reservations"].sum() == len(df_clean)
)

print(f"All Validation Checks Passed: {validation_passed}")

Temporal Continuity Validation:
Expected Calendar Days: 549
Observed Calendar Days: 549
Missing Calendar Dates: 0
Duplicated Dates: 0
Chronologically Sorted: True

Target Integrity Validation:
Missing Target Values: 0
Negative Target Values: 0
Zero-Reservation Days: 0
Target Data Type: int64

Overall Validation:
All Validation Checks Passed: True


#### Key Observations
- **Complete Temporal Coverage:** All 549 calendar dates are represented, with no missing or duplicated dates.
- **Valid Forecasting Target:** Daily scheduled reservation counts contain no missing, negative, or zero values and retain an integer data type.
- **Successful Validation:** All temporal continuity and target integrity checks passed, confirming that the aggregated dataset is structurally suitable for time-series preparation.

#### Saving the Prepared Daily Forecasting Dataset

In [10]:
daily_file = processed_dir / "daily_reservations.csv"

assert validation_passed, "Daily dataset validation failed."

daily_df.to_csv(daily_file, index=False)

saved_daily_df = pd.read_csv(
    daily_file,
    parse_dates=["arrival_datetime"]
)

print("Daily Forecasting Dataset Saved Successfully")
print(f"File Path: {daily_file}")
print(f"File Exists: {daily_file.exists()}")

print("\nSaved Dataset Validation:")
print(f"Dataset Shape: {saved_daily_df.shape}")
print(f"Total Reservations: {saved_daily_df['scheduled_reservations'].sum():,}")
print(f"Date Range: {saved_daily_df['arrival_datetime'].min().date()} to {saved_daily_df['arrival_datetime'].max().date()}")
print(f"Matches Original Daily Dataset: {saved_daily_df.equals(daily_df)}")

Daily Forecasting Dataset Saved Successfully
File Path: ..\data\processed\daily_reservations.csv
File Exists: True

Saved Dataset Validation:
Dataset Shape: (549, 2)
Total Reservations: 36,238
Date Range: 2017-07-01 to 2018-12-31
Matches Original Daily Dataset: True


#### Key Observations
- **Successful Artifact Creation:** The validated daily forecasting dataset has been saved as `data/processed/daily_reservations.csv`, containing 549 daily observations.
- **Data Consistency Verified:** The saved dataset matches the prepared DataFrame exactly, preserving all 36,238 valid reservations and the complete date range.

#### Defining Chronological Training and Testing Periods

In [11]:
train_ratio = 0.80
forecast_horizon = 7

split_index = int(len(daily_df) * train_ratio)

train_df = daily_df.iloc[:split_index].copy()
test_df = daily_df.iloc[split_index:].copy()

print("Chronological Split Summary:")
print(f"Total Observations: {len(daily_df):,}")
print(f"Training Observations: {len(train_df):,}")
print(f"Testing Observations: {len(test_df):,}")
print(f"Forecast Horizon: {forecast_horizon} days")

print("\nTraining Period:")
print(f"Start Date: {train_df['arrival_datetime'].min().date()}")
print(f"End Date: {train_df['arrival_datetime'].max().date()}")

print("\nTesting Period:")
print(f"Start Date: {test_df['arrival_datetime'].min().date()}")
print(f"End Date: {test_df['arrival_datetime'].max().date()}")

print("\nSplit Validation:")
print(
    "Chronological Separation:",
    train_df["arrival_datetime"].max()
    < test_df["arrival_datetime"].min()
)

print(
    "Total Observations Preserved:",
    len(train_df) + len(test_df) == len(daily_df)
)

print(
    "Overlapping Dates:",
    len(
        set(train_df["arrival_datetime"])
        & set(test_df["arrival_datetime"])
    )
)

Chronological Split Summary:
Total Observations: 549
Training Observations: 439
Testing Observations: 110
Forecast Horizon: 7 days

Training Period:
Start Date: 2017-07-01
End Date: 2018-09-12

Testing Period:
Start Date: 2018-09-13
End Date: 2018-12-31

Split Validation:
Chronological Separation: True
Total Observations Preserved: True
Overlapping Dates: 0


#### Key Observations
- **Chronological Training Period:** The training dataset contains 439 daily observations (80%), covering July 1, 2017, through September 12, 2018.
- **Independent Testing Period:** The testing dataset contains 110 daily observations (20%), covering September 13, 2018, through December 31, 2018.
- **Split Integrity:** All 549 observations are preserved, with no overlapping dates between training and testing periods.

#### Saving Chronological Training and Testing Datasets

In [12]:
train_file = processed_dir / "train_daily.csv"
test_file = processed_dir / "test_daily.csv"

train_df.to_csv(train_file, index=False)
test_df.to_csv(test_file, index=False)

saved_train_df = pd.read_csv(
    train_file,
    parse_dates=["arrival_datetime"]
)

saved_test_df = pd.read_csv(
    test_file,
    parse_dates=["arrival_datetime"]
)

print("Chronological Datasets Saved Successfully")

print("\nTraining Dataset:")
print(f"File: {train_file}")
print(f"File Exists: {train_file.exists()}")
print(f"Shape: {saved_train_df.shape}")
print(f"Matches Original: {saved_train_df.equals(train_df)}")

print("\nTesting Dataset:")
print(f"File: {test_file}")
print(f"File Exists: {test_file.exists()}")
print(f"Shape: {saved_test_df.shape}")
print(f"Matches Original: {saved_test_df.equals(test_df)}")

print("\nFinal Split Validation:")
print(
    "All Observations Preserved:",
    len(saved_train_df) + len(saved_test_df) == len(daily_df)
)

print(
    "Chronological Separation:",
    saved_train_df["arrival_datetime"].max()
    < saved_test_df["arrival_datetime"].min()
)

Chronological Datasets Saved Successfully

Training Dataset:
File: ..\data\processed\train_daily.csv
File Exists: True
Shape: (439, 2)
Matches Original: True

Testing Dataset:
File: ..\data\processed\test_daily.csv
File Exists: True
Shape: (110, 2)
Matches Original: False

Final Split Validation:
All Observations Preserved: True
Chronological Separation: True


#### Investigating Saved Dataset Consistency

In [13]:
print("Original Testing Data Types:")
print(test_df.dtypes.to_string())

print("\nReloaded Testing Data Types:")
print(saved_test_df.dtypes.to_string())

print("\nIndex Comparison:")
print(f"Original Index: {test_df.index[:5].tolist()}")
print(f"Reloaded Index: {saved_test_df.index[:5].tolist()}")

print("\nColumn Comparison:")
print(f"Columns Match: {test_df.columns.equals(saved_test_df.columns)}")

print("\nValue Comparison (Ignoring Index):")
print(
    "All Values Match:",
    test_df.reset_index(drop=True).equals(saved_test_df)
)

print("\nDetailed DataFrame Comparison:")
try:
    pd.testing.assert_frame_equal(
        test_df.reset_index(drop=True),
        saved_test_df
    )
    print("DataFrames Match: True")
except AssertionError as error:
    print(f"DataFrames Match: False\n{error}")

Original Testing Data Types:
arrival_datetime          datetime64[us]
scheduled_reservations             int64

Reloaded Testing Data Types:
arrival_datetime          datetime64[us]
scheduled_reservations             int64

Index Comparison:
Original Index: [439, 440, 441, 442, 443]
Reloaded Index: [0, 1, 2, 3, 4]

Column Comparison:
Columns Match: True

Value Comparison (Ignoring Index):
All Values Match: True

Detailed DataFrame Comparison:
DataFrames Match: True


#### Key Observations
- **Successful Dataset Preservation:** Both chronological datasets were saved successfully, containing 439 training observations and 110 testing observations.
- **Index Difference Identified:** The testing dataset retained its original index starting at 439, while the reloaded CSV starts at 0. All column names, data types, and values match exactly.
- **Split Integrity Confirmed:** All 549 observations are preserved, and the training and testing periods remain chronologically separated.

#### Performing Final Data Preparation Validation

In [15]:
cleaned_check = pd.read_csv(
    processed_dir / "cleaned_reservations.csv",
    parse_dates=["arrival_datetime"]
)

daily_check = pd.read_csv(
    processed_dir / "daily_reservations.csv",
    parse_dates=["arrival_datetime"]
)

train_check = pd.read_csv(
    processed_dir / "train_daily.csv",
    parse_dates=["arrival_datetime"]
)

test_check = pd.read_csv(
    processed_dir / "test_daily.csv",
    parse_dates=["arrival_datetime"]
)

combined_check = pd.concat(
    [train_check, test_check],
    ignore_index=True
)

validation_results = {
    "All Artifacts Exist": all(
        path.exists()
        for path in [
            cleaned_file,
            daily_file,
            train_file,
            test_file
        ]
    ),
    "Reservation Totals Match": (
        len(cleaned_check)
        == daily_check["scheduled_reservations"].sum()
    ),
    "Daily Dataset Matches Splits": combined_check.equals(daily_check),
    "Training Dataset Matches": train_check.equals(
        train_df.reset_index(drop=True)
    ),
    "Testing Dataset Matches": test_check.equals(
        test_df.reset_index(drop=True)
    ),
    "No Overlapping Dates": (
        train_check["arrival_datetime"].max()
        < test_check["arrival_datetime"].min()
    ),
    "No Missing Daily Dates": (
        len(daily_check)
        == len(
            pd.date_range(
                daily_check["arrival_datetime"].min(),
                daily_check["arrival_datetime"].max(),
                freq="D"
            )
        )
    ),
    "No Missing Target Values": (
        daily_check["scheduled_reservations"].notna().all()
    )
}

print("Final Data Preparation Validation:")
for check, result in validation_results.items():
    print(f"{check}: {result}")

print("\nOverall Status:")
print(
    "All Validation Checks Passed:",
    all(validation_results.values())
)

print("\nSaved Artifacts:")
for path in [cleaned_file, daily_file, train_file, test_file]:
    print(f"{path.name}: {path.exists()}")

Final Data Preparation Validation:
All Artifacts Exist: True
Reservation Totals Match: True
Daily Dataset Matches Splits: True
Training Dataset Matches: True
Testing Dataset Matches: True
No Overlapping Dates: True
No Missing Daily Dates: True
No Missing Target Values: True

Overall Status:
All Validation Checks Passed: True

Saved Artifacts:
cleaned_reservations.csv: True
daily_reservations.csv: True
train_daily.csv: True
test_daily.csv: True


#### Key Observations
- **Artifact Integrity:** All four processed datasets were saved successfully and passed their respective consistency checks.
- **Chronological Consistency:** The training and testing datasets reconstruct the complete 549-day forecasting series without overlapping dates or missing observations.
- **Data Preparation Readiness:** Reservation totals, target values, and chronological boundaries have been validated, establishing a consistent foundation for feature engineering and forecasting.

## Data Preparation Summary

The data preparation process transformed the original hotel reservation dataset into a validated daily time-series dataset suitable for forecasting scheduled reservation volumes.

The preparation workflow included:

- **Data Quality Validation:** Verified missing values, duplicate records, booking identifiers, and arrival date validity.
- **Invalid Date Handling:** Excluded 37 records containing invalid arrival dates, retaining 36,238 valid reservations.
- **Reservation-Level Cleaning:** Created and saved a cleaned, chronologically ordered reservation-level dataset.
- **Daily Aggregation:** Aggregated valid reservations by scheduled arrival date, producing 549 daily observations between July 1, 2017, and December 31, 2018.
- **Temporal Validation:** Confirmed complete calendar coverage, unique dates, and valid nonnegative reservation counts.
- **Chronological Splitting:** Allocated 439 observations (80%) to training and 110 observations (20%) to testing without randomization or overlapping dates.
- **Artifact Preservation:** Saved and verified four reusable datasets for subsequent feature engineering, model development, and evaluation.

### Prepared Artifacts

| Artifact | Description |
|---|---|
| `cleaned_reservations.csv` | Cleaned reservation-level dataset containing 36,238 records |
| `daily_reservations.csv` | Complete daily forecasting dataset containing 549 observations |
| `train_daily.csv` | Chronological training dataset containing 439 observations |
| `test_daily.csv` | Chronological testing dataset containing 110 observations |

### Forecasting Considerations

The project targets a **7-day forecasting horizon**, with the final chronological testing period reserved for evaluating predictive performance.

Feature engineering and model development must preserve temporal ordering, prevent future target information from entering historical predictors, and apply time-aware validation.

Because the daily series is derived from retrospective reservation records grouped by scheduled arrival date, the dataset does not establish which reservation totals were observable at earlier forecast origins. Consequently, forecasting results will be interpreted as a retrospective time-series modeling exercise rather than a validated real-time operational forecasting system.

**The completed preparation process provides a consistent, validated, and reusable foundation for developing and evaluating daily hotel reservation forecasting models.**